<a href="https://colab.research.google.com/github/olajidechris/CollabProjects/blob/main/Cleaning_And_Generating_Short_Titles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

olajidechris_productdata_csv_path = kagglehub.dataset_download('olajidechris/productdata-csv')

print('Data source import complete.')


### Stage 1: Cleaning And Generating Short Titles

In [ ]:
#Setup Gdrive file download extention
#!pip install -y gdown
#!gdown --id 1z3Io4xtC2FMif2lskLbDDcRe5OTjoi4l

# Move file from working to input
#import shutil
#shutil.move('/kaggle/working/productdata.xlsx', '/kaggle/input/productdata.xlsx')

In [ ]:
# Get input data file pathsin the
paths_input_files = []
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
# check file paths
        paths_input_files.append(os.path.join(dirname, filename))

print(paths_input_files)

# Import libraries for Analysis
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np # linear algebra
import matplotlib.pyplot as plt # data visualisations
import seaborn as sns # more data visualisations
import re

# suppress warnings due to version changes
import warnings
warnings.filterwarnings('ignore')

display("Setup Completed")

['/kaggle/input/productdata-csv/productdata.xlsx - Sheet1.csv.crdownload']


'Setup Completed'

Steps to Complete the Task
1. Dataset Familiarization
<br>Download and review the dataset to understand its structure and key variables.
<br>Identify significant variables for cleaning and for creating the short_title feature: The ['PRODUCTTYPEID'] column looks like it may benefit from
<br>Check column names for clarity and consistency: Columns appear to use a mix of UPPERCASE, UPPER_CASE_WITH_UNDERSCORES, and CamelCase style formats.

In [ ]:
# Load dataset csv file
df = pd.read_csv(paths_input_files[0])

display(df.info(),
        df.head(),
        df.tail()
       )

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3847 entries, 0 to 3846
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   PRODUCTID      3847 non-null   int64  
 1   TITLE          3847 non-null   object 
 2   BULLET_POINTS  2256 non-null   object 
 3   DESCRIPTION    1703 non-null   object 
 4   PRODUCTTYPEID  3669 non-null   float64
 5   ProductLength  3669 non-null   float64
dtypes: float64(2), int64(1), object(3)
memory usage: 180.5+ KB


None

,PRODUCTID,TITLE,BULLET_POINTS,DESCRIPTION,PRODUCTTYPEID,ProductLength
0,1925202,ArtzFolio Tulip Flowers Blackout Curtain for D...,[LUXURIOUS & APPEALING: Beautiful custom-made ...,NaN,1650.0,2125.980000
1,2673191,Marks & Spencer Girls' Pyjama Sets T86_2561C_N...,"[Harry Potter Hedwig Pyjamas (6-16 Yrs),100% c...",NaN,2755.0,393.700000
2,2765088,PRIKNIK Horn Red Electric Air Horn Compressor ...,"[Loud Dual Tone Trumpet Horn, Compatible With ...","Specifications: Color: Red, Material: Aluminiu...",7537.0,748.031495
3,1594019,ALISHAH Women's Cotton Ankle Length Leggings C...,[Made By 95%cotton and 5% Lycra which gives yo...,AISHAH Women's Lycra Cotton Ankel Leggings. Br...,2996.0,787.401574
4,283658,The United Empire Loyalists: A Chronicle of th...,NaN,NaN,6112.0,598.424000


,PRODUCTID,TITLE,BULLET_POINTS,DESCRIPTION,PRODUCTTYPEID,ProductLength
3842,67078,Occupational Health and Safety: International ...,NaN,NaN,6232.0,598.000
3843,653247,Le duel introuvable,NaN,NaN,1.0,539.369
3844,1783479,"ArtToFrames 27x39 Inch Black Picture Frame, Th...",[Comes with 060 Plexi Glass and a a wire hangi...,This black satin frame is our favorite gallery...,1590.0,2900.000
3845,770603,The Ascension of Isaiah: translated from the E...,NaN,NaN,6104.0,550.000
3846,112393,Crossings: Selected Travel Writings,NaN,NaN,12320.0,600.000


In [ ]:
with pd.option_context('display.max_colwidth', None):
    print(df['TITLE'].head(2), "\n\n")
    print(df['BULLET_POINTS'].head(2), "\n\n")
    print(df['DESCRIPTION'].dropna().head(2))


0    ArtzFolio Tulip Flowers Blackout Curtain for Door, Window & Room | Eyelets & Tie Back | Canvas Fabric | Width 4.5feet (54inch) Height 5 feet (60 inch); Set of 2 PCS
1                                                                                                             Marks & Spencer Girls' Pyjama Sets T86_2561C_Navy Mix_9-10Y
Name: TITLE, dtype: object 


0    [LUXURIOUS & APPEALING: Beautiful custom-made curtains to decorate any home or office | Includes inbuilt tieback to hold the curtain | Completely finished and ready to hang on walls & windows,MATERIAL: Luxurious & versatile fabric with a natural finish | High colour fastness | State-of-the-art digital printing ensures colour consistency and prevents any fading | Eyelets; Cotton Canvas; Width 4.5feet (54inch) | Multicolour | PACKAGE: 2 Room Curtains Eyelets | SIZE: Height 5 feet (60 inch); SET OF 2 PCS,BLACKOUT CURTAIN: 100% opaque & heavy premium cotton canvas fabric | Tight knitted, long life & durable fabric | Print

2. Initial Data Exploration
<br>Conduct a high-level review of the dataset using descriptive statistics.


In [ ]:
display(df.select_dtypes(include=np.number).describe(),
        df.select_dtypes(exclude=np.number).describe()
       )

,PRODUCTID,PRODUCTTYPEID,ProductLength
count,3.847000e+03,3669.000000,3669.000000
mean,1.456557e+06,3932.736986,1150.529020
std,8.666684e+05,3970.908660,2665.897894
min,1.303000e+03,0.000000,1.000000
25%,6.922785e+05,154.000000,507.873000
50%,1.441218e+06,2879.000000,640.000000
75%,2.214798e+06,6337.000000,1023.622046
max,2.999397e+06,13330.000000,96000.000000


,TITLE,BULLET_POINTS,DESCRIPTION
count,3847,2256,1703
unique,3541,2116,1609
top,Exacompta A3 Vega Opaque Pvc Display Book Land...,"[Good quality and Suitable to use.,This Produc...",This Case is Made up of Hard Polycarbonate Pla...
freq,3,14,14


<br>Identify data quality issues such as missing values, duplicates, and inconsistent formats using Pandas and NumPy.

In [ ]:
count_null_by_col = df.apply(lambda x :sum(x.isnull()))
unique_rows_by_freq = df.value_counts(dropna=False).reset_index(name='freq')
multiple_entries = unique_rows_by_freq.query('freq > 1.0')

display(count_null_by_col,"Number of null values in each column","",
        len(unique_rows_by_freq),"Number of unique entries","",
        multiple_entries['freq'].describe(),"Statistical data on multiple entry frequency","",
        multiple_entries.head(),
        multiple_entries.tail()
       )

PRODUCTID           0
TITLE               0
BULLET_POINTS    1591
DESCRIPTION      2144
PRODUCTTYPEID     178
ProductLength     178
dtype: int64

'Number of null values in each column'

''

3630

'Number of unique entries'

...

count    217.0
mean       2.0
std        0.0
min        2.0
25%        2.0
50%        2.0
75%        2.0
max        2.0
Name: freq, dtype: float64

'Statistical data on multiple entry frequency'

''

,PRODUCTID,TITLE,BULLET_POINTS,DESCRIPTION,PRODUCTTYPEID,ProductLength,freq
0,1706369,HOMEIDEAS 100% Blackout Curtains 52 X 63 Inch ...,NaN,NaN,NaN,NaN,2
1,1673378,Fossil Gen 5 Carlyle Stainless Steel Touchscre...,[Smartwatches powered with wear OS by Google w...,NaN,12426.0,500.000,2
2,354537,Dream Sequence in Red,NaN,NaN,99.0,600.000,2
3,106994,ME AND MY SHADOWS: A Family Memoir,NaN,NaN,21.0,425.000,2
4,1390057,Game of Thrones - Season 1-7 [Blu-ray] [2017] ...,NaN,NaN,3.0,700.786,2


,PRODUCTID,TITLE,BULLET_POINTS,DESCRIPTION,PRODUCTTYPEID,ProductLength,freq
212,719366,Raunacht: Das Osterreichische Fantasy-Epos,NaN,NaN,102.0,600.000000,2
213,2934559,QuecyÂ® Women's Sexy Asymmetrical Hem Scoop Ne...,"[88% Polyester, 12% Elastane, Pull On closure,...",Quecy Women's Sexy Asymmetrical Hem Scoop Neck...,3077.0,1338.582676,2
214,1501832,adidas Men's Predator 18+ FG Firm Ground Socce...,NaN,NaN,NaN,NaN,2
215,2683584,BANARASI SILK PALACE Women's Shubh Vasttram Va...,[Package-: This Comfortable Saree Comes With O...,"Quintessentially Indian, the saree is an outfi...",2917.0,590.551180,2
216,770603,The Ascension of Isaiah: translated from the E...,NaN,NaN,6104.0,550.000000,2


3. Data Cleaning Process
<br>Handling Missing Values: Identify and evaluate columns with missing data.
<br>Removing Duplicate Entries: Locate and remove duplicate records to ensure data accuracy.
<br>Standardize column names (e.g., sales_amount instead of SalesAmount).
<br>Verifying Data Accuracy: Check critical variables for anomalies (e.g., negative prices or invalid ratings).

In [ ]:
#unique_rows.iloc[:, 0:-1].applymap(type())
unique_rows_by_freq.apply(lambda x: x.dropna().apply(lambda y: type(y)).unique())

# replace nulls
cleaned = unique_rows_by_freq.iloc[:, 0:-1].replace(to_replace={
                                                                    'BULLET_POINTS': np.NaN,
                                                                    'DESCRIPTION': np.NaN,
                                                                    'PRODUCTTYPEID': np.NaN,
                                                                    'ProductLength': np.NaN},
                                                             value={
                                                                    'BULLET_POINTS': "",
                                                                    'DESCRIPTION': "",
                                                                    'PRODUCTTYPEID': None,
                                                                    'ProductLength': None})

# change columns names to camelcase
cleaned.columns = ['ProductID', 'Title', 'BulletPoints',
                   'Description', 'ProductTypeID', 'ProductLength']

cleaned['ShortTitle'] = cleaned['Title']
cleaned.head(3)

,ProductID,Title,BulletPoints,Description,ProductTypeID,ProductLength,ShortTitle
0,1706369,HOMEIDEAS 100% Blackout Curtains 52 X 63 Inch ...,,,None,None,HOMEIDEAS 100% Blackout Curtains 52 X 63 Inch ...
1,1673378,Fossil Gen 5 Carlyle Stainless Steel Touchscre...,[Smartwatches powered with wear OS by Google w...,,12426.0,500.0,Fossil Gen 5 Carlyle Stainless Steel Touchscre...
2,354537,Dream Sequence in Red,,,99.0,600.0,Dream Sequence in Red


4. Creating the short_title Feature
<br>Objective: Generate a concise version of product titles that retains essential information for SEO and readability.
<br>Steps: Analyze original product titles to identify key components (e.g., product name, category, attributes).
<br>Remove redundant phrases or words (e.g., "includes," "set of," "features").
<br>Limit titles to 30–50 characters, focusing on essential details and keywords.
<br>Implementation Example:
<br>Original Title: "Tulip Flowers Blackout Curtain for Door, Window & Room | Eyelets & Tie Back | Canvas Fabric | Set of 2 PCS"
<br>Short Title: "Tulip Blackout Curtain - 2 PCS"
<br>Original Title: "Marks & Spencer Girls' Pyjama Sets T86_2561C_Navy Mix_9-10Y"
<br>Short Title: "Girls' Navy Pyjama Set - 9-10Y"

In [ ]:
print(
    cleaned['ShortTitle'].apply(lambda s: clean_qty_color(s)).head(),"\n",
    cleaned['Title'].head())

print(
    cleaned['ShortTitle'].apply(lambda s: len(s)).head(),"\n",
    cleaned['Title'].head())

0    (HOMEIDEAS Blackout Curtains  Length 2 Panels ...
1    (Fossil Gen 5 Carlyle Stainless Steel Touchscr...
2                                Dream Sequence in Red
3                   ME AND MY SHADOWS: A Family Memoir
4    (Game of Thrones - Season [Blu-ray] [] [Region...
Name: ShortTitle, dtype: object 
 0    HOMEIDEAS 100% Blackout Curtains 52 X 63 Inch ...
1    Fossil Gen 5 Carlyle Stainless Steel Touchscre...
2                                Dream Sequence in Red
3                   ME AND MY SHADOWS: A Family Memoir
4    Game of Thrones - Season 1-7 [Blu-ray] [2017] ...
Name: Title, dtype: object


In [ ]:

redun_pattern = r'(?<.{30,})(include|feature|with|length|width|\||\[|\-|:).*'
set_pattern = r'(?<![a-zA-Z])(sets|set|pck|pcks|packs|pack|combos|combo|bundles|bundle|lots|lot)(?![a-zA-Z])'
pcs_pattern = r'(?<![a-zA-Z])(pieces|piece|pcs|pc|items|item)(?![a-zA-Z])'

qty_pattern = r'(Pcs|Set of *\d+)|(Pcs *\d+)|(\d+ *Pcs|Set)|(\d+[*\-]*\d+ *[a-zA-Z]*)'
color_pattern = r'((?<![a-zA-Z])(red|blue|Grey|green|yellow|purple|orange|brown|white|brown|black)(?![a-zA-Z]))'

def format_str(s):
    s = re.sub(" +", ' ', s) # remove multiple spaces
    s = re.sub(" X|x *", "*", s) # remove spaces around 'x'
    s = re.sub(" *\* *", "*", s) # remove spaces around 'x'
    s = re.sub("\s*,\s*", ',', s)   # replace underscore with space
    s = re.sub("\d*\s*%", '', s)   # remove number before '%'
    s = re.sub(" & ", '&', s)   # remove space before and after '&'
    s = re.sub('_', ' ', s)   # replace underscore with space
    s = re.sub("('s)|(')", ' ', s)   # delete single quote or 's
    s = re.sub("\-+", '-', s) # remove multiple dashes
    s = re.sub(" +", ' ', s) # remove multiple spaces
    if s[0] == " ": s = s[1:]
    if s[-1] == " ": s = s[:-1]

    return s

def remove_redun(s):
    if len(s) <= 50:
        return s
    return re.sub(redun_pattern, '', s) # remove redun_pattern

def clean_qty_color(s):
    # qty_pattern color_pattern to extract qty and color
    if len(s) <= 50:
        return (s, '', '')
    s = format_str(s) # reformat and standardize string
    # if match qty
    s = re.sub(set_pattern, ' Set', s, re.IGNORECASE) # rewrite set quantities spaces
    s = re.sub(pcs_pattern, ' Pcs', s, re.IGNORECASE) # rewrite pieces quantities spaces
    s = format_str(s) # reformat and standardize string
    if len(s) <= 50:
        return s
    colors, quantities = "", ""
    match_qty = re.search(qty_pattern, s, re.IGNORECASE)
    if match_qty:
        quantities = match_qty.group()
        s = re.sub(qty_pattern, '', s, re.IGNORECASE) # remove redundant_qty words
    # if match color
    match_colors = re.search(color_pattern, s, re.IGNORECASE)
    if match_colors:
        colors = match_colors.group()
        s = re.sub(color_pattern, '', s, re.IGNORECASE) # remove color words
    return (s, quantities, colors)

def shorten_str(s, quantities, colors):
    words = re.split(r"[|\'\":,_/\s]", s)  # split s into words
    #for w
    if len(words) > 1: #if multiple words
        intervening_txt = words[-1]  # assign last word of s as intervening_txt
        s = s.replace(intervening_txt, '') # remove intervening_txt from s
        s = ' '.join(words[:-1]) # rejoin remaining words into str
    else:
        intervening_txt = ""

    s = f'{s}-{intervening_txt}' if intervening_txt else s # add intervening_txt to end of s
    return s


In [ ]:
def create_short_title(s):
    s = remove_redun(s)
    s = format_str(s) # reformat and standardize string
    s, quantities, colors = get_qty_color(s)    # get qty color
    parts = re.split(r"[|:,_/(...)]", s) # split text by punctuation into list
    words = re.split(r"[|\'\":,_/\s]", s) # split text by space/punctuation into list
    #unique_words =
    extra_text = parts
    intervening_txt = ""
    if len(extra_text)>1:
        extra_text, intervening_txt = extra_text[:-1], shorten_str(extra_text[-1])

    short_title = f"{extra_text}-{intervening_txt}" if (intervening_txt!="") else ' '.join(extra_text) # remove last str in list if no attributes
    short_title = f"{short_title}-{quantities}-{colors}" # join short_title, qty, color as short_title
    short_title = format_str(short_title)

    rough_desc = f"{parts[0]}-{quantities}-{colors}" # join rough_desc, qty, color as rough_desc
    rough_desc = format_str(rough_desc)

    if len(short_title) > 50:
        if len(short_title) > len(rough_desc):
            short_title = rough_desc
    elif len(short_title) < 20:
        short_title = rough_desc
    return short_title[:50]  # reformat and standardize string

In [ ]:
print(create_short_title(cleaned.iloc[0, 1]))
print(create_short_title(cleaned.iloc[1, 1]))

HOMEIDEAS Blackout Curtains 52*63 Inch-
Fossil Gen 5 Carlyle Stainless Steel Touchscreen M


In [ ]:
import re
from textblob import TextBlob

def describe_str(s):

    blob_object = TextBlob(s)
    return(blob_object.tags)

print(describe_str(cleaned.iloc[0, 1]))
print(get_qty_color(cleaned.iloc[0, 1]))

match_qty = re.search('(\d+ Pcs)|(\d+[*\-]*d+)',format_str(cleaned.iloc[0, 1]), re.IGNORECASE)
match_qty.group()


[('100', 'CD'), ('%', 'NN'), ('Blackout', 'NNP'), ('Curtains', 'NNP'), ('52', 'CD'), ('X', 'NNP'), ('63', 'CD'), ('Inch', 'NNP'), ('Length', 'NNP'), ('2', 'CD'), ('Panels', 'NNP'), ('Light', 'NNP'), ('Grey', 'NNP'), ('Total', 'NNP'), ('Room', 'NNP'), ('Darkening', 'NNP'), ('Curtains', 'NNP'), ('for', 'IN'), ('Bedroom', 'NNP'), ('Thermal', 'NNP'), ('Grommet', 'NNP'), ('Double', 'NNP'), ('Layers', 'NNP'), ('Thick', 'NNP'), ('Soundproof', 'NNP'), ('Curtains', 'NNP'), ('Energy', 'NNP'), ('Saving', 'VBG')]
('HOMEIDEAS Blackout Curtains 52*63 Inch Length 2 Panels Light  Total Room Darkening Curtains for Bedroom,Thermal Grommet Double Layers Thick Soundproof Curtains Energy Saving', '', 'Grey')


AttributeError: 'NoneType' object has no attribute 'group'

CC coordinating conjunction
CD cardinal digit
DT determiner
EX existential there (like: “there is” … think of it like “there exists”)
FW foreign word
IN preposition/subordinating conjunction
JJ adjective ‘big’
JJR adjective, comparative ‘bigger’
JJS adjective, superlative ‘biggest’
LS list marker 1)
MD modal could, will
NN noun, singular ‘desk’
NNS noun plural ‘desks’
NNP proper noun, singular ‘Harrison’
NNPS proper noun, plural ‘Americans’
PDT predeterminer ‘all the kids’
POS possessive ending parent‘s
PRP personal pronoun I, he, she
PRP$ possessive pronoun my, his, hers
RB adverb very, silently,
RBR adverb, comparative better
RBS adverb, superlative best
RP particle give up
TO to go ‘to‘ the store.
UH interjection errrrrrrrm
VB verb, base form take
VBD verb, past tense took
VBG verb, gerund/present participle taking
VBN verb, past participle taken
VBP verb, sing. present, non-3d take
VBZ verb, 3rd person sing. present takes
WDT wh-determiner which
WP wh-pronoun who, what
WP$ possessive wh-pronoun whose
WRB wh-adverb where, when

In [ ]:
example_1 = ["Marks & Spencer Girls' Pyjama Sets T86_2561C_Navy Mix_9-10Y",
             "Girls' Navy Pyjama Set - 9-10Y"]
example_2 = ["Tulip Flowers Blackout Curtain for Door, Window & Room | Eyelets & Tie Back | Canvas Fabric | Set of 2 PCS",
             "Tulip Blackout Curtain - 2 PCS"]
test = "Avima Women's Printed Pyjama| Cotton Pyjama for Women Night Wear|Women Night Wear|Soft Cotton Night Women Pyjama (Black(Multi), L)"

print(create_short_title(test))

print(describe_str(example_1[1]))
print(describe_str(example_2[1]),"\n")

print(create_short_title(example_1[0]))
print(create_short_title(example_2[0]),"\n")


Avima Women Printed Pyjama-Black
[('Girls', 'NNP'), ("'", 'POS'), ('Navy', 'NNP'), ('Pyjama', 'NNP'), ('Set', 'NNP'), ('9-10Y', 'JJ')]
[('Tulip', 'NN'), ('Blackout', 'NNP'), ('Curtain', 'NNP'), ('2', 'CD'), ('PCS', 'NNS')] 

Marks&Spencer Girls Pyjama T86 2561C Navy Mi*9-10Y
Tulip Flowers Blackout Curtain for Door-Set of 2- 



In [ ]:
s = cleaned.iloc[:, 0:2].copy()
s['Title'] = s['Title'].apply(lambda x: create_short_title(x))

with pd.option_context('display.max_colwidth', None):
    display(s.head(50), s.tail(50))

,ProductID,Title
0,1706369,HOMEIDEAS Blackout Curtains 52*63 Inch -
1,1673378,Fossil Gen 5 Carlyle Stainless Steel Touchscreen M
2,354537,Dream Sequence in -Red
3,106994,[ ME AND MY SHADOWS ]- A Family-Memoir-
4,1390057,Game of Thrones - Season 1-7 [Blu-ray] [2017] [Reg
5,107333,Carl Pops Up--
6,1801723,PNW Components The Coast Gravel Handlebar--
7,2292711,Grow Forward Bamboo Bowl and Plate - 4 Bamboo Plat
8,2749965,PosterHub Pink Floyd The Wall Poster Matte Finish
9,112393,[ Crossings ]- Selected Travel-Writings-


,ProductID,Title
3580,997808,NFL Kansas City Chiefs Team Shield Automobile Refl
3581,997926,A*iom Memory Solutionlc 8gb Ddr3-1333 Ecc Rdimm Ta
3582,999198,Dell Vostro 3555 Laptop Screen 15--
3583,999906,e-Book eBook sleeve for Kindle--
3584,1002333,LCD Panel--
3585,1002451,Champion Men Jersey Short -
3586,1005463,Detroit--
3587,1008258,The Mountain Bigfoot Adult T-Shirt--Brown
3588,1008592,[ Frends Taylor ]-Silver and-Black
3589,1009449,3dRose db 70207 1 Giant Panda Bears--


In [ ]:
from textblob import TextBlob

def lemmatize_title(title):
    sent = TextBlob(title)
    tag_dict = {"J": 'a', "N": 'n', "V": 'v', "R": 'r'}
    words_tags = [(w, tag_dict.get(pos[0], 'n')) for w, pos in sent.tags]
    lemma_list = [wd.lemmatize(tag) for wd, tag in words_tags]
    lemmatized_title = ' '.join(lemma_list)
    return lemma_list

# Lemmatize
sentence = "the bats saw the cats with stripes hanging upside down by their feet"
lemma_list = pos_tagger(sentence)
print(lemmatized_sentence)


In [ ]:
display(cleaned['Title'][cleaned.Title.str.len() > 404].describe(),
        cleaned.Description[cleaned.Description.str.len() > 1024].describe())

count       0
unique      0
top       NaN
freq      NaN
Name: Title, dtype: object

count                                                   326
unique                                                  325
top       An original piece of mobile phone cover(s)/cas...
freq                                                      2
Name: Description, dtype: object

    # change text to title case and remove duplicates
    s = [i.title() for i in s.split()]
    s = ' '.join(pd.Series(s).value_counts(sort=False).index.to_list())
    return s

    # check for ':' indicating book and other titles
def check_1(s, char):
    if len(s) > 50:
        if s.find(char) != -1:
            return s.split(char)[-1]
    return s

with pd.option_context('display.max_colwidth', None):
    print(cleaned['Title'].apply(lambda x: check_1(x, ":")).head(10))

import pandas as pd
import re
from textblob import TextBlob

def extract_product_info(text):
    # change text to title case
    s = ' '.join(set(title(s.split())))
    
    # Remove content inside parentheses or brackets
    text_clean = re.sub(r'[\(\[].*?[\)\]]', '', text)

    # Remove brand names (the brand list was updated and expanded as needed)
    brands = [
        'Marks & Spencer', 'ALISHAH', 'SEGOVIA', 'Tulip Flowers',
        'Kenneth Cole REACTION', 'BaronHong', 'Vans', 'Red Tape',
        'Hot Wheels', 'Lowepro'    ]
    
    for brand in brands:
        text_clean = re.sub(re.escape(brand), '', text_clean, flags=re.IGNORECASE).strip()

    # Remove model numbers or codes
    text_clean = re.sub(r'\b[A-Z0-9._%-]+(?:_[A-Z0-9._%-]+)+\b', '', text_clean)
    text_clean = re.sub(r'\bT\d+[_-]\d+\w*\b', '', text_clean, flags=re.IGNORECASE)
    text_clean = re.sub(r'_+', ' ', text_clean)  # Replace multiple underscores with space

    # Remove excess whitespace and special characters
    text_clean = re.sub(r'[^\w\s]', ' ', text_clean)
    text_clean = ' '.join(text_clean.split())

    # Create a TextBlob object
    blob = TextBlob(text_clean)

    # Initialize variables
    product_name = ''
    category = ''
    attributes = []

    # Extract noun phrases and POS tags
    noun_phrases = [np.lower() for np in blob.noun_phrases]
    pos_tags = blob.tags

    # Define category keywords
    category_keywords = ['curtain', 'pyjama', 'leggings', 'bottle', 'jacket', 'loafers', 'shoes', 'binder', 'camera', 'toy', 'underwear', 'books', 'car']

    # Identify category
    for keyword in category_keywords:
        if keyword in text_clean.lower():
            category = keyword.title()
            break

    # If category not found in keywords, use POS tags (looking for nouns)
    if not category:
        for word, tag in pos_tags:
            if tag.startswith('NN'):  # Noun tags
                category = word.title()
                break

    # Identify product name
    # Assume product name is the first noun phrase that includes the category
    for np in noun_phrases:
        if category.lower() in np:
            product_name = np.title()
            break

    # Fallback to the first noun phrase if product name is still empty
    if not product_name and noun_phrases:
        product_name = noun_phrases[0].title()

    # Additional attributes extraction
    size_pattern = r'\b(\d+(\.\d+)?\s*(ml|l|oz|inches|inch|ft|feet|cm|mm|m|kg|g|lbs|xl|l|m|s|xs|xxl|xxxl|\d+y|\d+-\d+y))\b'
    color_pattern = r'\b(black|white|red|green|blue|yellow|pink|grey|gray|purple|maroon|navy|silver|gold|orange|brown|beige|ivory)\b'
    quantity_pattern = r'\b(pack of \d+|set of \d+|combo of \d+|[\d]+\s*pcs?|[\d]+\s*pieces?)\b'
    material_pattern = r'\b(cotton|leather|silk|stainless steel|suede|canvas|metal|plastic|wooden|ceramic)\b'

    # Search for patterns in the text
    size_matches = re.findall(size_pattern, text_clean, flags=re.IGNORECASE)
    color_matches = re.findall(color_pattern, text_clean, flags=re.IGNORECASE)
    quantity_matches = re.findall(quantity_pattern, text_clean, flags=re.IGNORECASE)
    material_matches = re.findall(material_pattern, text_clean, flags=re.IGNORECASE)

    # Add found attributes
    for match in size_matches:
        attributes.append(match[0])
    for match in color_matches:
        attributes.append(match)
    for match in quantity_matches:
        attributes.append(match)
    for match in material_matches:
        attributes.append(match)

    # Include adjectives from POS tags as additional attributes
    for word, pos in pos_tags:
        if pos in ['JJ', 'JJR', 'JJS'] and word.lower() not in attributes:
            attributes.append(word)

    # Remove duplicates and join attributes
    attributes = list(set(attributes))
    attributes_str = ', '.join(attributes).title()

    # Return the extracted information
    return pd.Series([product_name, category, attributes_str])

# Apply the function to the DataFrame
df[['Product Name', 'Category', 'Attributes']] = df['title'].apply(extract_product_info)

# Display the results
print(df[['title', 'Product Name', 'Category', 'Attributes']])


In [ ]:
import re
def clean_str(s):
    s = s.title()
    s = ' '.join(set(s.split()))
    s = re.sub(r'(\d+[(\%)(mm)])', ' ', s.title())
    s = s.split('You')[0]
    s = re.sub(r'(\W+)', ' ', s)
    s = re.sub(r'\W+[a-zA-Z0-9]\W+', ' ', s)
    s = re.sub(r'(\[+[a-zA-Z0-9]+\])+', ' ', s)
    s = re.sub(r'( [a-zA-Z0-9]+ [(Or )(Of )])', ' ', s)
#    s = re.sub(r'(\W+[a-zA-Z0-9]+\W+([Oo]r)\W+[a-zA-Z0-9]+\W+)', ' ', s)
    s = re.sub(r'((Inch)|(Length)|(Width))\W+\d+', ' ', s)
    s = re.sub(r'([^a-zA-Z0-9\-]+\W*)', ' ', s)
    s = re.sub(r'\W+\-\W+', '\-', s)
    s = re.sub(r'\W+[a-zA-Z]\W+', ' ', s)
    s = re.sub(r'(The\W)|(For\W)|(With\W)', ' ', s)
    s = re.sub(r'(Include\W)|(To\W)|(In\W)', ' ', s)
    s = re.sub(r'[(Women\W)(Woman\W)]{2,}', 'Women ', s)
    s = re.sub(r'[(Men\W)(Man\W)]{2,}', 'Men ', s)
    s = re.sub(r'[(Women\W)(Men\W)(Unisex\W)]{2,}', '', s)
    s = re.sub(r'(\d+\W+\d+)', ' ', s)
    s = re.sub(r'\W+[a-zA-Z0-9]\W+', ' ', s)
    s = re.sub(r'(\W+)', ' ', s)
    s = ' '.join(set(s.split()))

    return s



,ProductID
count,3.630000e+03
mean,1.458613e+06
std,8.681912e+05
min,1.303000e+03
25%,6.919762e+05
50%,1.442706e+06
75%,2.217074e+06
max,2.999397e+06


,ProductID,Title
0,1706369,Soundprf DoublGrey Total PlBlackout R Thick Hd52 Grt Light 63 BedrLayerDarkg Ergy Savg CurtaTherl
1,1673378,CarlylBlack Rat TouchscrFtw4024 And GpFl GNotificatStl SrtphStalHrt Srtwatch Spker
2,354537,DrSequcRed
3,106994,ry AndShadowFly
4,1390057,SAll 2017 FrBlu RegDutGTaThrImport And Ray Includ
5,107333,Carlp Pops
6,1801723,FlarDeg Rch 20 10565Cpt52 Ct 48CHdlebar Gravel Pnw Drop
7,2292711,Dhwasher BbForward Bowl DhFrdly Grow BowlDrwarFrBdegradablReusable Fd Set BpPlatArctic SafEcPlatAnd
8,2749965,12 ll X18 Pterhub PkFhulticolorttPter Prt Paper HP076 Floyd
9,112393,Selected CrgTravelritgs


,ProductID,Title
3580,997808,Nfl TShld KaChfCity Reflector Autbile
3581,997926,Ecc ADdr Rd ry TSolutlc 8Gb Cplt
3582,999198,ScrVtro Left Dell Hd 3555 Bott15 LaptopgLed
3583,999906,SlvBk Touch Kdl FirPaperwhitKeybrd KdlEbk
3584,1002333,Lcd Pl
3585,1002451,PocketChpSll Jery ShortBlack
3586,1005463,Detrt
3587,1008258,Sll ShirtBigft T AdultuntaBrow
3588,1008592,And FrdTaylor Black Silver
3589,1009449,Gt ChAs07 Pd3DrCrvatDb ChP0374 By Brlg Oxford Drawg PetBk


In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

def generate_summary(text):
    inputs = tokenizer.encode("Category, Attributes " + text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(inputs, max_length=10, min_length=1, length_penalty=2.0, num_beams=4, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [ ]:
test = cleaned.Title.head().apply(lambda x: generate_summary(x))
test

0               Curtains are 100%
1            Fossil Gen 5 Carlyle
2    Summarize: Dream Sequence in
3             Summarize:ME AND MY
4     Game of Thrones - Season 1-
Name: Title, dtype: object

In [ ]:
cleaned.head()

Steps to Complete the Task


5. Documentation and Reporting
Prepare a professional report documenting the data cleaning and title optimization process:
Introduction: Briefly describe the dataset and task objectives.
Data Cleaning: Summarize issues identified and the cleaning steps taken.
Short Title Creation: Explain the methodology and examples of optimized titles.
Clean Dataset Overview: Highlight key statistics and improvements in the dataset.
Include screenshots or visualizations illustrating the impact of data cleaning and title optimization.
6. Review and Submission
Proofread and format the report for clarity and professionalism.
Submit the cleaned dataset, including the short_title column, and the report.
Deliverables
Cleaned Dataset: Updated dataset free of missing values, duplicates, and inconsistencies, with the short_title feature added.
Technical Report: A professional document including:
Introduction and objectives.
Detailed steps for data cleaning and title optimization.
Summary of improvements, with examples of original and short titles.